# 01 - AOI & Data Acquisition

Defines the 8 wheat-growing state AOIs (FAO GAUL), Rabi season windows, and acquires **Sentinel-1 SAR**, **Sentinel-2 optical** (cloud-masked), and **MODIS NDVI/LST**. Also demonstrates local ingestion of **Resourcesat-2 AWiFS** GeoTIFFs and **IMD** gridded NetCDF.

> Prerequisite: `pip install -r ../requirements.txt` and a Google Earth Engine account (`ee.Authenticate()`).

In [1]:
import sys, yaml
sys.path.append('..')
from src import gee_utils, io_utils

with open('../config/config.yaml') as f:
    cfg = yaml.safe_load(f)
STATES = cfg['states']
STATES

E:\codes\Gitlab project\wheat-crop-monitoring\venv\lib\site-packages\google\api_core\_python_version_support.py:255: FutureWarning: You are using a Python version (3.10.11) which Google will stop supporting in new releases of google.api_core once it reaches its end of life (2026-10-04). Please upgrade to the latest Python version, or at least Python 3.11, to continue receiving updates for google.api_core past that date.
  warnings.warn(message, FutureWarning)


['Uttar Pradesh',
 'Madhya Pradesh',
 'Punjab',
 'Haryana',
 'Rajasthan',
 'Bihar',
 'Gujarat',
 'Maharashtra']

In [2]:
import ee
gee_utils.init_ee()  # first run triggers ee.Authenticate()
states_fc = gee_utils.get_states_fc(STATES)
aoi = states_fc.geometry()
print('States loaded:', states_fc.size().getInfo())

States loaded: 8


## Rabi season window

Sowing starts ~1 November; harvest ends ~30 April. Early-season optical data is cloud-limited, hence the **SAR-first** strategy.

In [3]:
SEASON_YEAR = 2023  # Rabi 2023-24
start = f'{SEASON_YEAR}-11-01'
end = f'{SEASON_YEAR + 1}-04-30'

s1 = gee_utils.get_s1_collection(aoi, start, end)
s2 = gee_utils.get_s2_collection(aoi, start, end, cfg['sensors']['sentinel2']['max_cloud_prob'])
print('Sentinel-1 scenes:', s1.size().getInfo())
print('Sentinel-2 scenes (cloud-masked):', s2.size().getInfo())

Sentinel-1 scenes: 893
Sentinel-2 scenes (cloud-masked): 13431


In [4]:
modis_ndvi = gee_utils.get_modis_ndvi(aoi, start, end)
modis_lst = gee_utils.get_modis_lst(aoi, start, end)
print('MODIS NDVI composites:', modis_ndvi.size().getInfo())
print('MODIS LST composites:', modis_lst.size().getInfo())

MODIS NDVI composites: 12
MODIS LST composites: 23


In [5]:
import geemap
m = geemap.Map(center=[26.5, 78.0], zoom=5)
m.addLayer(states_fc.style(color='black', fillColor='00000000'), {}, 'States')
m.addLayer(s2.median().select(['B4', 'B3', 'B2']), {'min': 0, 'max': 0.3}, 'S2 median RGB')
m.addLayer(s1.median().select('VH'), {'min': -25, 'max': -5}, 'S1 VH median', False)
m

Map(center=[26.5, 78.0], controls=(WidgetControl(options=['position', 'transparent_bg'], position='topright', …

## Local data: AWiFS / LISS-III and IMD

**Caution:** AWiFS 56 m resolution causes mixed-pixel anomalies on smallholder plots - use it for wide-swath state coverage, Sentinel-2 for plot-level work.

In [6]:
# AWiFS GeoTIFF ingestion (uncomment with real paths)
# arr, profile = io_utils.load_awifs_geotiff('path/to/awifs_scene.tif')
# ndvi = io_utils.awifs_ndvi(arr[1], arr[2])  # band2=red, band3=NIR

# IMD gridded NetCDF (0.25 deg rainfall / 1 deg temperature)
# rain = io_utils.load_imd_netcdf('path/to/imd_rainfall.nc', var='RAINFALL')
print('Replace paths above with real AWiFS / IMD files when available.')

Replace paths above with real AWiFS / IMD files when available.


## NASA POWER Weather Data Acquisition

Fetches daily **temperature**, **precipitation**, and **solar radiation** from the [NASA POWER API](https://power.larc.nasa.gov/) for representative coordinates in each of the 8 wheat-growing states. Data is saved to `data/nasa_power/` for use in **Notebook 04** yield forecasting.

Parameters fetched:
- `T2M` – Temperature at 2m (°C)
- `T2M_MAX` – Max temperature (°C)
- `T2M_MIN` – Min temperature (°C)
- `PRECTOTCORR` – Corrected precipitation (mm/day)
- `ALLSKY_SFC_SW_DWN` – Surface shortwave downward irradiance (MJ/m²/day)

In [7]:
import requests, time, os, pathlib
import pandas as pd

# Representative coordinates for major wheat-growing districts in each state
STATE_COORDS = {
    'Punjab':          {'lat': 30.90, 'lon': 75.80},  # Ludhiana
    'Haryana':         {'lat': 29.15, 'lon': 76.40},  # Karnal
    'Uttar_Pradesh':   {'lat': 27.50, 'lon': 80.00},  # Lucknow belt
    'Madhya_Pradesh':  {'lat': 23.50, 'lon': 78.00},  # Hoshangabad
    'Rajasthan':       {'lat': 29.90, 'lon': 73.88},  # Sri Ganganagar
    'Bihar':           {'lat': 25.60, 'lon': 84.00},  # Rohtas
    'Gujarat':         {'lat': 23.00, 'lon': 72.00},  # Ahmedabad
    'Maharashtra':     {'lat': 19.50, 'lon': 75.50},  # Ahmednagar
}

PARAMS = 'T2M,T2M_MAX,T2M_MIN,PRECTOTCORR,ALLSKY_SFC_SW_DWN'
START = 20150101
END   = 20231231
COMMUNITY = 'AG'  # Agricultural
OUT_DIR = pathlib.Path('../data/nasa_power')
OUT_DIR.mkdir(parents=True, exist_ok=True)

BASE_URL = 'https://power.larc.nasa.gov/api/temporal/daily/point'

for state, coord in STATE_COORDS.items():
    out_file = OUT_DIR / f'POWER_{state}_{START}_{END}.csv'
    if out_file.exists():
        print(f'  ⏭ {state}: already downloaded')
        continue

    print(f'  ⬇ Fetching {state} ({coord["lat"]}, {coord["lon"]})...', end=' ')
    url = (
        f'{BASE_URL}?parameters={PARAMS}'
        f'&community={COMMUNITY}'
        f'&longitude={coord["lon"]}'
        f'&latitude={coord["lat"]}'
        f'&start={START}&end={END}'
        f'&format=CSV'
    )
    try:
        resp = requests.get(url, timeout=120)
        resp.raise_for_status()
        with open(out_file, 'w', encoding='utf-8') as f:
            f.write(resp.text)
        print('✓')
    except Exception as e:
        print(f'✗ {e}')

    time.sleep(2)  # be polite to NASA API

# Summary
fetched = list(OUT_DIR.glob('*.csv'))
print(f'\n✓ {len(fetched)} state weather files in {OUT_DIR}')
for fp in sorted(fetched):
    print(f'  {fp.name} ({fp.stat().st_size / 1024:.0f} KB)')

  ⏭ Punjab: already downloaded
  ⏭ Haryana: already downloaded
  ⏭ Uttar_Pradesh: already downloaded
  ⏭ Madhya_Pradesh: already downloaded
  ⏭ Rajasthan: already downloaded
  ⏭ Bihar: already downloaded
  ⏭ Gujarat: already downloaded
  ⏭ Maharashtra: already downloaded

✓ 8 state weather files in ..\data\nasa_power
  POWER_Bihar_20150101_20231231.csv (125 KB)
  POWER_Gujarat_20150101_20231231.csv (125 KB)
  POWER_Haryana_20150101_20231231.csv (124 KB)
  POWER_Madhya_Pradesh_20150101_20231231.csv (125 KB)
  POWER_Maharashtra_20150101_20231231.csv (125 KB)
  POWER_Punjab_20150101_20231231.csv (124 KB)
  POWER_Rajasthan_20150101_20231231.csv (124 KB)
  POWER_Uttar_Pradesh_20150101_20231231.csv (124 KB)


## NASA POWER Weather Data Acquisition

Fetches daily **temperature**, **precipitation**, and **solar radiation** from the [NASA POWER API](https://power.larc.nasa.gov/) for representative coordinates in each of the 8 wheat-growing states. Data is saved to `data/nasa_power/` for use in **Notebook 04** yield forecasting.

Parameters fetched:
- `T2M` – Temperature at 2m (°C)
- `T2M_MAX` – Max temperature (°C)
- `T2M_MIN` – Min temperature (°C)
- `PRECTOTCORR` – Corrected precipitation (mm/day)
- `ALLSKY_SFC_SW_DWN` – Surface shortwave downward irradiance (MJ/m²/day)

In [ ]:
import requests, time, os, pathlib
import pandas as pd

# Representative coordinates for major wheat-growing districts in each state
STATE_COORDS = {
    'Punjab':          {'lat': 30.90, 'lon': 75.80},  # Ludhiana
    'Haryana':         {'lat': 29.15, 'lon': 76.40},  # Karnal
    'Uttar_Pradesh':   {'lat': 27.50, 'lon': 80.00},  # Lucknow belt
    'Madhya_Pradesh':  {'lat': 23.50, 'lon': 78.00},  # Hoshangabad
    'Rajasthan':       {'lat': 29.90, 'lon': 73.88},  # Sri Ganganagar
    'Bihar':           {'lat': 25.60, 'lon': 84.00},  # Rohtas
    'Gujarat':         {'lat': 23.00, 'lon': 72.00},  # Ahmedabad
    'Maharashtra':     {'lat': 19.50, 'lon': 75.50},  # Ahmednagar
}

PARAMS = 'T2M,T2M_MAX,T2M_MIN,PRECTOTCORR,ALLSKY_SFC_SW_DWN'
START = 20150101
END   = 20231231
COMMUNITY = 'AG'  # Agricultural
OUT_DIR = pathlib.Path('../data/nasa_power')
OUT_DIR.mkdir(parents=True, exist_ok=True)

BASE_URL = 'https://power.larc.nasa.gov/api/temporal/daily/point'

for state, coord in STATE_COORDS.items():
    out_file = OUT_DIR / f'POWER_{state}_{START}_{END}.csv'
    if out_file.exists():
        print(f'  ⏭ {state}: already downloaded')
        continue

    print(f'  ⬇ Fetching {state} ({coord["lat"]}, {coord["lon"]})...', end=' ')
    url = (
        f'{BASE_URL}?parameters={PARAMS}'
        f'&community={COMMUNITY}'
        f'&longitude={coord["lon"]}'
        f'&latitude={coord["lat"]}'
        f'&start={START}&end={END}'
        f'&format=CSV'
    )
    try:
        resp = requests.get(url, timeout=120)
        resp.raise_for_status()
        with open(out_file, 'w', encoding='utf-8') as f:
            f.write(resp.text)
        print('✓')
    except Exception as e:
        print(f'✗ {e}')

    time.sleep(2)  # be polite to NASA API

# Summary
fetched = list(OUT_DIR.glob('*.csv'))
print(f'\n✓ {len(fetched)} state weather files in {OUT_DIR}')
for fp in sorted(fetched):
    print(f'  {fp.name} ({fp.stat().st_size / 1024:.0f} KB)')